# TurboQuant KV Cache Benchmark
Test TurboQuant 3-bit and 4-bit KV cache quantization on WorldSim models
Compare: q8_0 (baseline) vs turbo4 vs turbo3
Models: 0.8B, 2B, 4B FT v3.1


## 1. Build TurboQuant llama.cpp Fork

In [ ]:
import json
import os
import random
import re
import shutil
import subprocess
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import yaml

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'config' / 'generation.yaml').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

GGUF_DIR = REPO_ROOT / 'artifacts' / 'gguf'
CONFIG_DIR = REPO_ROOT / 'config'

ORIGINAL_LLAMA = REPO_ROOT / 'tools' / 'llama.cpp'
TURBO_LLAMA = REPO_ROOT / 'tools' / 'llama-cpp-turboquant'
TURBO_SERVER = TURBO_LLAMA / 'build' / 'bin' / 'llama-server'
TURBO_BENCH = TURBO_LLAMA / 'build' / 'bin' / 'llama-bench'
ORIGINAL_SERVER = ORIGINAL_LLAMA / 'build' / 'bin' / 'llama-server'

if not TURBO_SERVER.exists():
    print('=== Cloning TurboQuant fork ===', flush=True)
    if not TURBO_LLAMA.exists():
        subprocess.run(
            [
                'git', 'clone', '--depth', '1',
                'https://github.com/TheTom/llama-cpp-turboquant.git',
                str(TURBO_LLAMA),
            ],
            check=True,
        )

    print('=== Building with CUDA ===', flush=True)
    build_dir = TURBO_LLAMA / 'build'
    build_dir.mkdir(exist_ok=True)

    subprocess.run(
        ['cmake', '..', '-DGGML_CUDA=ON', '-DCMAKE_BUILD_TYPE=Release'],
        cwd=str(build_dir),
        check=True,
        capture_output=True,
        text=True,
    )

    nproc = os.cpu_count() or 8
    subprocess.run(
        ['cmake', '--build', '.', '--config', 'Release', '-j', str(nproc)],
        cwd=str(build_dir),
        check=True,
        capture_output=True,
        text=True,
    )
    print('Build complete', flush=True)
else:
    print(f'TurboQuant already built: {TURBO_SERVER}')

assert TURBO_SERVER.exists(), f'TurboQuant server not found: {TURBO_SERVER}'
print(f'  Server: {TURBO_SERVER}')
if TURBO_BENCH.exists():
    print(f'  Bench: {TURBO_BENCH}')


def load_yaml(path: Path):
    with path.open(encoding='utf-8') as handle:
        return yaml.safe_load(handle)


situations = load_yaml(REPO_ROOT / 'config/situations.yaml')['situations']
personalities = load_yaml(REPO_ROOT / 'config/personalities.yaml')['personalities']
emotions = load_yaml(REPO_ROOT / 'config/emotions.yaml')['emotions']
social_situations = load_yaml(REPO_ROOT / 'config/social_situations.yaml')['social_situations']
group_situations = load_yaml(REPO_ROOT / 'config/group_situations.yaml')['group_situations']
trade_scenarios = load_yaml(REPO_ROOT / 'config/trade_scenarios.yaml')['trade_scenarios']
stress_sources = load_yaml(REPO_ROOT / 'config/stress_sources.yaml')['stress_sources']
deception_scenarios = load_yaml(REPO_ROOT / 'config/deception_scenarios.yaml')['deception_scenarios']
rumor_scenarios = load_yaml(REPO_ROOT / 'config/rumor_scenarios.yaml')['rumor_scenarios']
trauma_scenarios = load_yaml(REPO_ROOT / 'config/trauma_scenarios.yaml')['trauma_scenarios']
negotiation_scenarios = load_yaml(REPO_ROOT / 'config/negotiation_scenarios.yaml')['negotiation_scenarios']
culture_scenarios = load_yaml(REPO_ROOT / 'config/culture_scenarios.yaml')['culture_scenarios']
group_dissent_scenarios = load_yaml(REPO_ROOT / 'config/group_dissent_scenarios.yaml')['group_dissent_scenarios']

TEMPERAMENTS = [
    {'id': 'choleric', 'ns': 0.8, 'ha': 0.2, 'rd': 0.5, 'p': 0.7, 'keywords': 'bold, impulsive, dominant'},
    {'id': 'melancholic', 'ns': 0.2, 'ha': 0.8, 'rd': 0.6, 'p': 0.4, 'keywords': 'cautious, anxious, detail-oriented'},
    {'id': 'sanguine', 'ns': 0.6, 'ha': 0.3, 'rd': 0.8, 'p': 0.3, 'keywords': 'sociable, warm, optimistic'},
    {'id': 'phlegmatic', 'ns': 0.3, 'ha': 0.5, 'rd': 0.4, 'p': 0.8, 'keywords': 'calm, patient, persistent'},
]

MODELS = {}
model_configs = [
    ('ft_08b', 'worldsim-v31-qwen3.5-0.8b-q4_k_m.gguf', '0.8B FT v3.1'),
    ('ft_2b', 'worldsim-v31-qwen3.5-2b-q4_k_m.gguf', '2B FT v3.1'),
    ('ft_4b', 'worldsim-v31-qwen3.5-4b-q4_k_m.gguf', '4B FT v3.1'),
]

print('\n=== Model Inventory ===')
for key, filename, label in model_configs:
    path = GGUF_DIR / filename
    if path.exists():
        MODELS[key] = {'path': path, 'label': label, 'size_mb': round(path.stat().st_size / 1e6)}
        print(f"  {label:.<25} {MODELS[key]['size_mb']} MB")
    else:
        print(f'  {label:.<25} NOT FOUND')

print(f'\n{len(MODELS)} models ready')


## 2. llama-bench Speed Test

In [ ]:
KV_CONFIGS = [
    {'name': 'q8_0 (baseline)', 'ctk': 'q8_0', 'ctv': 'q8_0'},
    {'name': 'turbo4', 'ctk': 'turbo4', 'ctv': 'turbo4'},
    {'name': 'turbo3', 'ctk': 'turbo3', 'ctv': 'turbo3'},
]

BENCH_RESULTS = []
bench_binary = TURBO_BENCH if TURBO_BENCH.exists() else None

if bench_binary:
    for model_key, model_info in MODELS.items():
        for kv in KV_CONFIGS:
            print(f"\n--- {model_info['label']} + {kv['name']} ---", flush=True)
            args = [
                str(bench_binary),
                '-m', str(model_info['path']),
                '-ngl', '99',
                '-fa', '1',
                '-ctk', kv['ctk'],
                '-ctv', kv['ctv'],
                '-t', '4',
                '-n', '128',
                '-p', '256',
                '-r', '3',
            ]
            try:
                result = subprocess.run(args, capture_output=True, text=True, timeout=120)
                output = result.stdout + result.stderr
                print(output[-500:] if len(output) > 500 else output)
                BENCH_RESULTS.append({
                    'model': model_key,
                    'model_label': model_info['label'],
                    'kv_config': kv['name'],
                    'ctk': kv['ctk'],
                    'output': output,
                    'returncode': result.returncode,
                })
            except subprocess.TimeoutExpired:
                print('  TIMEOUT')
                BENCH_RESULTS.append({
                    'model': model_key,
                    'model_label': model_info['label'],
                    'kv_config': kv['name'],
                    'ctk': kv['ctk'],
                    'output': 'TIMEOUT',
                    'returncode': -1,
                })
            except Exception as exc:
                print(f'  ERROR: {exc}')
                BENCH_RESULTS.append({
                    'model': model_key,
                    'model_label': model_info['label'],
                    'kv_config': kv['name'],
                    'ctk': kv['ctk'],
                    'output': f'ERROR: {exc}',
                    'returncode': -1,
                })
else:
    print('llama-bench not found - skipping raw speed test')
    print('Will test via llama-server API instead')


## 3. Server Helpers & Test Prompts

In [ ]:
import urllib.request

SERVER_PORT = 8090
SERVER_URL = f'http://127.0.0.1:{SERVER_PORT}'


def start_server(model_path, server_binary, ctk='q8_0', ctv='q8_0', ctx_size=2048, n_gpu_layers=99):
    args = [
        str(server_binary), '-m', str(model_path),
        '--host', '127.0.0.1', '--port', str(SERVER_PORT),
        '-c', str(ctx_size), '-np', '1', '-ngl', str(n_gpu_layers),
        '-fa', 'on', '--jinja', '--no-webui',
        '--chat-template-kwargs', '{"enable_thinking": false}',
        '-ctk', ctk, '-ctv', ctv,
    ]
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for attempt in range(120):
        time.sleep(0.5)
        try:
            resp = urllib.request.urlopen(f'{SERVER_URL}/health', timeout=2)
            data = json.loads(resp.read())
            if data.get('status') == 'ok':
                return proc
        except Exception:
            pass
        if proc.poll() is not None:
            stderr_out = proc.stderr.read().decode(errors='ignore')[-500:]
            raise RuntimeError(f'Server died: {stderr_out}')
    proc.kill()
    raise RuntimeError('Server startup timeout')


def stop_server(proc):
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()
            proc.wait()


def strip_think(text):
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def query_model(messages, response_format=None, max_tokens=256, temperature=0):
    payload = {
        'messages': messages,
        'max_tokens': max_tokens,
        'temperature': temperature,
        'stream': False,
    }
    if response_format:
        payload['response_format'] = response_format
    req = urllib.request.Request(
        f'{SERVER_URL}/v1/chat/completions',
        data=json.dumps(payload).encode(),
        headers={'Content-Type': 'application/json'},
    )
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read())
        content = data['choices'][0]['message']['content']
        return strip_think(content)
    except Exception as exc:
        return f'ERROR: {exc}'

rng = random.Random(42)

SYS_L3 = 'You are WorldSim logic assistant. Output JSON only. Follow [TEMP], [STRESS], [WORLD] context. Respect enum values and numeric ranges exactly.'
SYS_L4 = '너는 WorldSim 서사 도우미다. [TEMP], [STRESS], [WORLD]와 지정된 register를 반영해 JSON으로만 답하라.'
SYS_L5 = '너는 WorldSim 신탁 해석 도우미다. JSON으로만 답하라.'


def pick(items, n=1):
    return rng.sample(items, min(n, len(items)))


def temp_block(t):
    return f"NS={t['ns']} HA={t['ha']} RD={t['rd']} P={t['p']} type={t['id']}"


def personality_keywords(p):
    return ', '.join(p.get('keywords', []))


def json_schema(name: str, required: list[str], properties: dict) -> dict:
    return {
        'type': 'json_schema',
        'json_schema': {
            'name': name,
            'strict': True,
            'schema': {
                'type': 'object',
                'additionalProperties': False,
                'required': required,
                'properties': properties,
            },
        },
    }


emotion_ids = [e['id'] for e in emotions]
speaker_roles = ['elder', 'hunter', 'shaman', 'warrior', 'healer', 'gatherer', 'craftsman', 'chief', 'scout', 'observer']

b_schema = json_schema(
    'task_b',
    ['text_ko', 'text_en', 'register', 'emotion_expressed', 'intensity', 'mimetics', 'temperament_influence'],
    {
        'text_ko': {'type': 'string'},
        'text_en': {'type': 'string'},
        'register': {'type': 'string', 'enum': ['haera']},
        'emotion_expressed': {'type': 'string', 'enum': emotion_ids},
        'intensity': {'type': 'number'},
        'mimetics': {'type': 'array', 'items': {'type': 'string'}},
        'temperament_influence': {'type': 'string'},
    },
)
c_schema = json_schema(
    'task_c',
    ['speech_ko', 'speech_en', 'register', 'emotion_expressed', 'speaker_role', 'temperament_tone'],
    {
        'speech_ko': {'type': 'string'},
        'speech_en': {'type': 'string'},
        'register': {'type': 'string', 'enum': ['haera']},
        'emotion_expressed': {'type': 'string', 'enum': emotion_ids},
        'speaker_role': {'type': 'string', 'enum': speaker_roles},
        'temperament_tone': {'type': 'string'},
    },
)
e_schema_for = lambda option_count: json_schema(
    'task_e',
    ['action_id', 'confidence', 'hint', 'personality_reasoning', 'temperament_factor'],
    {
        'action_id': {'type': 'integer', 'enum': list(range(option_count))},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'hint': {'type': 'string'},
        'personality_reasoning': {'type': 'string', 'enum': ['novelty_seeking', 'harm_avoidance', 'reward_dependence', 'persistence']},
        'temperament_factor': {'type': 'string'},
    },
)
f_schema = json_schema(
    'task_f',
    ['emotion', 'intensity', 'cause', 'previous_emotion', 'transition_type', 'temperament_amplifier'],
    {
        'emotion': {'type': 'string', 'enum': emotion_ids},
        'intensity': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'cause': {'type': 'string'},
        'previous_emotion': {'type': 'string', 'enum': emotion_ids},
        'transition_type': {'type': 'string', 'enum': ['gradual', 'sudden', 'sustained']},
        'temperament_amplifier': {'type': 'string'},
    },
)
g_schema = json_schema(
    'task_g',
    ['interpretation_ko', 'interpretation_en', 'action_tendency', 'confidence', 'register', 'misinterpretation_type', 'temperament_bias'],
    {
        'interpretation_ko': {'type': 'string'},
        'interpretation_en': {'type': 'string'},
        'action_tendency': {'type': 'string', 'enum': ['mobilize', 'defend', 'wait', 'retreat', 'celebrate', 'mourn']},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'register': {'type': 'string', 'enum': ['haera', 'hao', 'hae']},
        'misinterpretation_type': {'type': 'string', 'enum': ['overconfident_literal', 'cautious_reversal', 'optimistic_expansion', 'passive_deferral', 'symbolic_abstraction']},
        'temperament_bias': {'type': 'string'},
    },
)
m_schema = json_schema(
    'task_m',
    ['decision_id', 'confidence', 'dissent_risk', 'reasoning', 'resource_commitment', 'timeline'],
    {
        'decision_id': {'type': 'integer'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'dissent_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'reasoning': {'type': 'string'},
        'resource_commitment': {'type': 'string', 'enum': ['food', 'tools', 'labor', 'weapons', 'none']},
        'timeline': {'type': 'string', 'enum': ['immediate', 'delayed', 'conditional']},
    },
)
o_schema = json_schema(
    'task_o',
    ['public_claim', 'private_truth', 'deception_style', 'lie_degree', 'detection_risk', 'confidence'],
    {
        'public_claim': {'type': 'string'},
        'private_truth': {'type': 'string'},
        'deception_style': {'type': 'string', 'enum': ['evasion', 'half_truth', 'outright_lie', 'exaggeration', 'omission']},
        'lie_degree': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'detection_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
    },
)
p_schema = json_schema(
    'task_p',
    ['retold_version', 'distortion_type', 'added_detail', 'dropped_detail', 'emotional_charge'],
    {
        'retold_version': {'type': 'string'},
        'distortion_type': {'type': 'string', 'enum': ['exaggeration', 'minimization', 'malicious_twist', 'emotional_coloring', 'detail_invention', 'source_confusion', 'faithful']},
        'added_detail': {'type': 'string'},
        'dropped_detail': {'type': 'string'},
        'emotional_charge': {'type': 'number', 'minimum': -1, 'maximum': 1},
    },
)
q_schema = json_schema(
    'task_q',
    ['trauma_response', 'behavioral_change', 'trigger_situation', 'intensity', 'duration', 'coping_mechanism'],
    {
        'trauma_response': {'type': 'string', 'enum': ['avoidance', 'overprotection', 'aggression', 'withdrawal', 'hypervigilance', 'ritual_coping', 'resilience']},
        'behavioral_change': {'type': 'string'},
        'trigger_situation': {'type': 'string'},
        'intensity': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'duration': {'type': 'string', 'enum': ['short_term', 'long_term', 'permanent']},
        'coping_mechanism': {'type': 'string'},
    },
)
r_schema = json_schema(
    'task_r',
    ['action', 'counter_give', 'counter_want', 'reasoning', 'emotional_state', 'walk_away_threshold'],
    {
        'action': {'type': 'string', 'enum': ['accept', 'counter_offer', 'reject', 'walk_away', 'stall', 'bluff']},
        'counter_give': {'type': 'string'},
        'counter_want': {'type': 'string'},
        'reasoning': {'type': 'string'},
        'emotional_state': {'type': 'string', 'enum': emotion_ids},
        'walk_away_threshold': {'type': 'number', 'minimum': 0, 'maximum': 1},
    },
)
t_schema = json_schema(
    'task_t',
    ['decision_id', 'confidence', 'dissent_risk', 'minority_position', 'minority_action', 'spark_event', 'reasoning', 'timeline'],
    {
        'decision_id': {'type': 'integer'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'dissent_risk': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'minority_position': {'type': 'integer'},
        'minority_action': {'type': 'string', 'enum': ['comply', 'grumble', 'passive_resist', 'splinter', 'coup_attempt']},
        'spark_event': {'type': 'string', 'enum': ['food_shortage', 'battle_loss', 'oracle_conflict', 'leader_death', 'betrayal', 'resource_discovery']},
        'reasoning': {'type': 'string'},
        'timeline': {'type': 'string', 'enum': ['immediate', 'delayed', 'conditional']},
    },
)

ALL_PROMPTS = []
pair_counter = 0

for sit in pick(situations, 5):
    emotion = rng.choice(emotions)
    intensity = rng.choice(emotion.get('intensities', [0.7]))
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        pair_counter += 1
        ALL_PROMPTS.append({
            'task': 'B',
            'pair_id': f"B-{pair_counter // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"B: {sit['id']} / {temp['id']} / {emotion['id']}",
            'system': SYS_L4,
            'user_prompt': (
                f"[TASK] B\n[TEMP]\n{temp_block(temp)}\n"
                f"[기질 이름] {temp['id']}\n"
                f"[기질 키워드] {temp['keywords']}\n"
                f"[인물 성격] {personality_keywords(pers)}\n"
                f"[지금 느끼는 것] {emotion['id']}:{intensity}\n"
                f"[STRESS] {round(rng.uniform(0.2, 0.9), 1)}\n"
                f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
                '[WORLD] default: 석기시대\n'
                '[출력 형식]\n'
                '{"text_ko": "해라체 2-3문장", "text_en": "English", "register": "haera", "emotion_expressed": "emotion", "intensity": 0.0, "mimetics": [], "temperament_influence": "str"}'
            ),
            'response_format': b_schema,
        })

for sit in pick(situations, 5):
    options = sit.get('action_options', sit.get('typical_actions', []))
    if not options:
        continue
    options_line = ' '.join(f"{i}:{o}" for i, o in enumerate(options))
    emotion = rng.choice(emotions)
    intensity = rng.choice(emotion.get('intensities', [0.7]))
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        pair_counter += 1
        ALL_PROMPTS.append({
            'task': 'E',
            'pair_id': f"E-{pair_counter // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"E: {sit['id']} / {temp['id']}",
            'system': SYS_L3,
            'user_prompt': (
                f"[TASK] E - Action Selection\n[TEMP]\n{temp_block(temp)}\n"
                f"[PERSONALITY] {temp['keywords']}\n"
                f"[EMOTION] {emotion['id']}:{intensity}\n"
                f"[STRESS] {round(rng.uniform(0.2, 0.9), 1)}\n"
                f"[SITUATION] {sit.get('desc', sit['id'])}\n"
                f"[OPTIONS]\n{options_line}\n"
                '[OUTPUT FORMAT]\n'
                '{"action_id": number, "confidence": 0.0-1.0, "hint": "English sentence", "personality_reasoning": "TCI axis", "temperament_factor": "snake_case"}'
            ),
            'response_format': e_schema_for(len(options)),
        })

for sit in pick(situations, 5):
    prev_emotion = rng.choice(emotions)
    for temp in [TEMPERAMENTS[2], TEMPERAMENTS[3]]:
        pers = rng.choice(personalities)
        ALL_PROMPTS.append({
            'task': 'F',
            'pair_id': f"F-{len(ALL_PROMPTS) // 2}",
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"F: {sit['id']} / {temp['id']} (prev={prev_emotion['id']})",
            'system': SYS_L3,
            'user_prompt': (
                f"[TASK] F - Emotion Judgment\n[TEMP]\n{temp_block(temp)}\n"
                f"[PERSONALITY] {temp['keywords']}\n"
                f"[CURRENT EMOTION] {prev_emotion['id']}:{rng.choice(prev_emotion.get('intensities', [0.5]))}\n"
                f"[SITUATION] {sit.get('desc', sit['id'])}\n"
                '[OUTPUT FORMAT]\n'
                f'{{"emotion": "one of 8", "intensity": 0.0-1.0, "cause": "sentence", "previous_emotion": "{prev_emotion["id"]}", "transition_type": "gradual|sudden|sustained", "temperament_amplifier": "str"}}'
            ),
            'response_format': f_schema,
        })

for sit in pick(situations, 5):
    emotion = rng.choice(emotions)
    register = pers_register = 'haera'
    for temp in [TEMPERAMENTS[0], TEMPERAMENTS[1]]:
        pers = rng.choice(personalities)
        ALL_PROMPTS.append({
            'task': 'C',
            'pair_id': None,
            'temperament_id': temp['id'],
            'personality_id': pers['id'],
            'scenario_id': sit['id'],
            'desc': f"C: {sit['id']} / {temp['id']} / {emotion['id']}",
            'system': SYS_L4,
            'user_prompt': (
                f"[TASK] C\n[TEMP]\n{temp_block(temp)}\n"
                f"[기질 이름] {temp['id']}\n"
                f"[인물 성격] {personality_keywords(pers)}\n"
                f"[지금 느끼는 것] {emotion['id']}:{rng.choice(emotion.get('intensities', [0.6]))}\n"
                f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
                '[역할] warrior\n'
                '[출력 형식]\n'
                '{"speech_ko": "해라체 대사", "speech_en": "English", "register": "haera", "emotion_expressed": "emotion", "speaker_role": "role", "temperament_tone": "str"}'
            ),
            'response_format': c_schema,
        })

for sit in pick(situations, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'G',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'oracle',
        'scenario_id': sit['id'],
        'desc': f"G: {sit['id']} / {temp['id']}",
        'system': SYS_L5,
        'user_prompt': (
            f"[TASK] G - Oracle Interpretation\n[TEMP]\n{temp_block(temp)}\n"
            f"[기질 이름] {temp['id']}\n"
            f"[인물 성격] {temp['keywords']}\n"
            '[ORACLE]\n"큰물이 밀려오기 전에 높은 곳으로 가라"\n'
            f"[벌어진 일] {sit.get('desc', sit['id'])}\n"
            '[역할] warrior\n'
            '[출력 형식]\n'
            '{"interpretation_ko": "해라체", "interpretation_en": "English", "action_tendency": "enum", "confidence": 0-1, "register": "haera", "misinterpretation_type": "enum", "temperament_bias": "str"}'
        ),
        'response_format': g_schema,
    })

for scenario in pick(deception_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    emotion = rng.choice(emotions)
    ALL_PROMPTS.append({
        'task': 'O',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"O: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] O - Deception\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] {emotion['id']}:{rng.choice(emotion.get('intensities', [0.5]))}\n"
            f"[TRUE_STATE] {scenario['true_state']}\n"
            f"[PUBLIC_GOAL] {scenario['public_goal']}\n"
            f"[TARGET] {scenario['target']}\n"
            f"[DETECTION_CONTEXT] {scenario['detection_context']}\n"
            '[OUTPUT FORMAT]\n'
            '{"public_claim": "str", "private_truth": "str", "deception_style": "enum", "lie_degree": 0-1, "detection_risk": 0-1, "confidence": 0-1}'
        ),
        'response_format': o_schema,
    })

for scenario in pick(rumor_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'P',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"P: {scenario['id']} / {temp['id']} / {scenario['reteller_relationship']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] P - Rumor\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] {rng.choice(emotions)['id']}:{round(rng.uniform(0.3, 0.8), 1)}\n"
            f"[ORIGINAL_FACT] {scenario['original_fact']}\n"
            f"[RETELLER_RELATIONSHIP] {scenario['reteller_relationship']}\n"
            '[OUTPUT FORMAT]\n'
            '{"retold_version": "str", "distortion_type": "enum", "added_detail": "str", "dropped_detail": "str", "emotional_charge": -1 to 1}'
        ),
        'response_format': p_schema,
    })

for scenario in pick(trauma_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'Q',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"Q: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] Q - Trauma\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] sadness:{round(rng.uniform(0.6, 0.95), 2)}\n"
            f"[TRAUMA_EVENT] {scenario['event']}\n"
            f"[TIME_SINCE] {scenario['time_since']}\n"
            f"[CURRENT_SITUATION] {scenario['current_situation']}\n"
            '[OUTPUT FORMAT]\n'
            '{"trauma_response": "enum", "behavioral_change": "str", "trigger_situation": "str", "intensity": 0-1, "duration": "enum", "coping_mechanism": "str"}'
        ),
        'response_format': q_schema,
    })

for scenario in pick(negotiation_scenarios, 10):
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'R',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"R: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] R - Negotiate\n[TEMP]\n{temp_block(temp)}\n"
            f"[PERSONALITY] {temp['keywords']}\n"
            f"[EMOTION] anticipation:{round(rng.uniform(0.3, 0.7), 1)}\n"
            f"[CONTEXT] {scenario['context']}\n"
            f"[OUR_RESOURCES] {scenario['our_resources']}\n"
            f"[THEIR_RESOURCES] {scenario['their_resources']}\n"
            f"[OFFER_HISTORY] {scenario['offer_history']}\n"
            f"[POWER_BALANCE] {scenario['power_balance']}\n"
            '[OPTIONS]\n0:accept 1:counter_offer 2:reject 3:walk_away 4:stall 5:bluff\n'
            '[OUTPUT FORMAT]\n'
            '{"action": "enum", "counter_give": "str", "counter_want": "str", "reasoning": "str", "emotional_state": "emotion", "walk_away_threshold": 0-1}'
        ),
        'response_format': r_schema,
    })

for scenario in pick(group_situations, 10):
    opts = scenario.get('options', [])
    opts_line = ' '.join(f"{o['id']}:{o.get('ko', o.get('desc', ''))}" for o in opts) if opts else '0:agree 1:disagree 2:delay'
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'M',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"M: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] M - Group Decision\n[TEMP]\nmean_NS=0.5 mean_HA=0.5 var=0.3 leader={temp['id']}\n"
            f"[GROUP_CONTEXT] {scenario.get('group_context', scenario.get('desc', scenario['id']))}\n"
            f"[SITUATION] {scenario.get('situation', scenario.get('desc', scenario['id']))}\n"
            f"[OPTIONS]\n{opts_line}\n"
            '[OUTPUT FORMAT]\n'
            '{"decision_id": number, "confidence": 0-1, "dissent_risk": 0-1, "reasoning": "str", "resource_commitment": "enum", "timeline": "enum"}'
        ),
        'response_format': m_schema,
    })

for scenario in pick(group_dissent_scenarios, 10):
    opts = scenario.get('options', [])
    opts_line = ' '.join(f"{o['id']}:{o['desc']}" for o in opts) if opts else '0:exile 1:trial 2:forgive'
    temp = rng.choice(TEMPERAMENTS)
    ALL_PROMPTS.append({
        'task': 'T',
        'pair_id': None,
        'temperament_id': temp['id'],
        'personality_id': 'random',
        'scenario_id': scenario['id'],
        'desc': f"T: {scenario['id']} / {temp['id']}",
        'system': SYS_L3,
        'user_prompt': (
            f"[TASK] T - Group Dissent\n[TEMP]\nmean_NS=0.5 mean_HA=0.5 var=0.4 leader={temp['id']}\n"
            f"[GROUP_CONTEXT] {scenario.get('group_context', '')}\n"
            f"[SITUATION] {scenario.get('situation', scenario['id'])}\n"
            f"[FACTION_HINT] {scenario.get('faction_hint', '')}\n"
            f"[OPTIONS]\n{opts_line}\n"
            '[OUTPUT FORMAT]\n'
            '{"decision_id": number, "confidence": 0-1, "dissent_risk": 0-1, "minority_position": number, "minority_action": "enum", "spark_event": "enum", "reasoning": "str", "timeline": "enum"}'
        ),
        'response_format': t_schema,
    })

task_counts = Counter(prompt['task'] for prompt in ALL_PROMPTS)
print('=== Test Prompt Summary ===')
for task, count in sorted(task_counts.items()):
    paired = sum(1 for prompt in ALL_PROMPTS if prompt['task'] == task and prompt.get('pair_id'))
    print(f"  Task {task}: {count} prompts ({paired} paired)")
print(f"  Total: {len(ALL_PROMPTS)} prompts × {len(MODELS)} models = {len(ALL_PROMPTS) * len(MODELS)} inferences")


## 4. Quality + Speed Benchmark

In [ ]:
RESULTS = []
SPEED_SUMMARY = []

test_binary = TURBO_SERVER

for model_key, model_info in MODELS.items():
    for kv in KV_CONFIGS:
        config_label = f"{model_info['label']} + {kv['name']}"
        print(f"\n{'=' * 60}")
        print(f'  {config_label}')
        print(f"{'=' * 60}", flush=True)

        try:
            proc = start_server(
                model_info['path'],
                test_binary,
                ctk=kv['ctk'],
                ctv=kv['ctv'],
            )
        except RuntimeError as exc:
            print(f'  FAILED: {exc}')
            SPEED_SUMMARY.append({
                'model': model_key,
                'model_label': model_info['label'],
                'kv_config': kv['name'],
                'status': 'FAILED',
                'error': str(exc),
            })
            continue

        time.sleep(1)
        latencies = []
        json_valid_count = 0

        try:
            for i, test in enumerate(ALL_PROMPTS):
                messages = [
                    {'role': 'system', 'content': test['system']},
                    {'role': 'user', 'content': test['user_prompt']},
                ]

                start_t = time.perf_counter()
                output = query_model(
                    messages,
                    response_format=test.get('response_format'),
                    max_tokens=256,
                    temperature=0,
                )
                latency = (time.perf_counter() - start_t) * 1000
                latencies.append(latency)

                json_valid = False
                parsed = None
                try:
                    parsed = json.loads(output)
                    json_valid = True
                    json_valid_count += 1
                except Exception:
                    pass

                RESULTS.append({
                    'model': model_key,
                    'model_label': model_info['label'],
                    'kv_config': kv['name'],
                    'config_label': config_label,
                    'task': test['task'],
                    'desc': test.get('desc', ''),
                    'pair_id': test.get('pair_id'),
                    'temperament_id': test.get('temperament_id'),
                    'output': output,
                    'parsed': parsed,
                    'json_valid': json_valid,
                    'latency_ms': round(latency),
                })

                if (i + 1) % 20 == 0:
                    avg_lat = sum(latencies[-20:]) / min(20, len(latencies))
                    print(
                        f"  [{i + 1}/{len(ALL_PROMPTS)}] avg_latency={avg_lat:.0f}ms json={json_valid_count}/{i + 1}",
                        flush=True,
                    )
        finally:
            stop_server(proc)
            time.sleep(2)

        avg_latency = sum(latencies) / len(latencies) if latencies else 0
        json_pct = json_valid_count / len(ALL_PROMPTS) * 100 if ALL_PROMPTS else 0

        SPEED_SUMMARY.append({
            'model': model_key,
            'model_label': model_info['label'],
            'kv_config': kv['name'],
            'avg_latency_ms': round(avg_latency),
            'json_pct': round(json_pct, 1),
            'total_prompts': len(ALL_PROMPTS),
            'status': 'OK',
        })

        print(f'  DONE avg={avg_latency:.0f}ms json={json_pct:.0f}%')

print(f'\nCollected {len(RESULTS)} total results')


## 5. Auto-Grade

In [ ]:
def auto_grade(result):
    grades = {}
    grades['json_valid'] = result['json_valid']
    placeholder_words = {'str', 'string', 'sentence', 'English', 'enum', 'number', '해라체', 'one of'}

    if result['parsed']:
        values = [str(v) for v in result['parsed'].values() if isinstance(v, str)]
        grades['not_placeholder'] = not any(v in placeholder_words for v in values)
    else:
        grades['not_placeholder'] = False

    if result['parsed']:
        str_fields = [v for v in result['parsed'].values() if isinstance(v, str)]
        avg_len = sum(len(s) for s in str_fields) / max(len(str_fields), 1)
        grades['text_richness'] = avg_len > 15
    else:
        grades['text_richness'] = False

    if result['parsed']:
        grades['numeric_valid'] = True
        for key in [
            'confidence',
            'intensity',
            'lie_degree',
            'detection_risk',
            'dissent_risk',
            'walk_away_threshold',
            'emotional_charge',
        ]:
            if key in result['parsed']:
                val = result['parsed'][key]
                if isinstance(val, (int, float)):
                    if key == 'emotional_charge':
                        grades['numeric_valid'] = -1 <= val <= 1
                    else:
                        grades['numeric_valid'] = 0 <= val <= 1
                else:
                    grades['numeric_valid'] = False
                break
    else:
        grades['numeric_valid'] = False

    grades['score'] = sum([
        grades.get('json_valid', False),
        grades.get('not_placeholder', False),
        grades.get('text_richness', False),
        grades.get('numeric_valid', False),
    ])
    return grades


for row in RESULTS:
    row['grades'] = auto_grade(row)

print(f'Graded {len(RESULTS)} results')


## 6. Results Summary

In [ ]:
print('=' * 100)
print('  TURBOQUANT KV CACHE BENCHMARK - RESULTS')
print('=' * 100)

print(f"\n  {'Configuration':<35} {'Avg Latency':>12} {'JSON%':>7} {'AvgScore':>10} {'!Placeholder':>13}")
print(f"  {'-' * 80}")

for model_key in ['ft_08b', 'ft_2b', 'ft_4b']:
    if model_key not in MODELS:
        continue
    for kv in KV_CONFIGS:
        config_label = f"{MODELS[model_key]['label']} + {kv['name']}"
        config_results = [
            r for r in RESULTS
            if r['model'] == model_key and r['kv_config'] == kv['name']
        ]

        if not config_results:
            print(f"  {config_label:<35} {'SKIPPED':>12}")
            continue

        n = len(config_results)
        avg_score = sum(r['grades']['score'] for r in config_results) / n
        json_pct = sum(1 for r in config_results if r['grades']['json_valid']) / n * 100
        np_pct = sum(1 for r in config_results if r['grades']['not_placeholder']) / n * 100
        avg_lat = sum(r['latency_ms'] for r in config_results) / n

        print(f'  {config_label:<35} {avg_lat:>10.0f}ms {json_pct:>6.0f}% {avg_score:>8.1f}/4 {np_pct:>12.0f}%')
    print()

print(f"\n  {'Model':<15} {'KV Config':<12} {'Latency Delta':>16} {'Score Delta':>12} {'JSON Delta':>11}")
print(f"  {'-' * 72}")

for model_key in ['ft_08b', 'ft_2b', 'ft_4b']:
    baseline = [r for r in RESULTS if r['model'] == model_key and r['kv_config'] == 'q8_0 (baseline)']
    if not baseline:
        continue

    base_lat = sum(r['latency_ms'] for r in baseline) / len(baseline)
    base_score = sum(r['grades']['score'] for r in baseline) / len(baseline)
    base_json = sum(1 for r in baseline if r['grades']['json_valid']) / len(baseline) * 100

    for kv in KV_CONFIGS[1:]:
        turbo = [r for r in RESULTS if r['model'] == model_key and r['kv_config'] == kv['name']]
        if not turbo:
            continue

        t_lat = sum(r['latency_ms'] for r in turbo) / len(turbo)
        t_score = sum(r['grades']['score'] for r in turbo) / len(turbo)
        t_json = sum(1 for r in turbo if r['grades']['json_valid']) / len(turbo) * 100

        lat_delta = t_lat - base_lat
        score_delta = t_score - base_score
        json_delta = t_json - base_json
        lat_pct = (lat_delta / base_lat) * 100 if base_lat else 0

        if score_delta >= -0.1 and json_delta >= -1:
            marker = 'OK'
        elif score_delta >= -0.3:
            marker = 'WARN'
        else:
            marker = 'BAD'

        print(
            f"  {marker:<4} {MODELS[model_key]['label']:<13} {kv['name']:<12} "
            f"{lat_delta:>+8.0f}ms ({lat_pct:>+5.1f}%) {score_delta:>+10.2f} {json_delta:>+9.1f}%"
        )


## 7. Personality Consistency

In [ ]:
print('=' * 80)
print('  PERSONALITY CONSISTENCY BY KV CONFIG')
print('=' * 80)

pairs = defaultdict(list)
for row in RESULTS:
    pair_id = row.get('pair_id')
    if pair_id:
        config_key = f"{row['model']}_{row['kv_config']}"
        pairs[(pair_id, config_key)].append(row)

consistency_by_config = defaultdict(lambda: {'same': 0, 'different': 0, 'total': 0})

for (pair_id, config_key), pair_rows in sorted(pairs.items()):
    if len(pair_rows) != 2:
        continue
    first, second = pair_rows
    if not first['parsed'] or not second['parsed']:
        continue

    task = first['task']
    different = False
    if task == 'E':
        different = first['parsed'].get('action_id') != second['parsed'].get('action_id')
    elif task == 'B':
        different = (
            first['parsed'].get('emotion_expressed') != second['parsed'].get('emotion_expressed')
            or abs(first['parsed'].get('intensity', 0) - second['parsed'].get('intensity', 0)) > 0.1
        )
    elif task == 'F':
        different = first['parsed'].get('emotion') != second['parsed'].get('emotion')

    consistency_by_config[config_key]['total'] += 1
    if different:
        consistency_by_config[config_key]['different'] += 1
    else:
        consistency_by_config[config_key]['same'] += 1

print(f"\n  {'Configuration':<35} {'Pairs':>6} {'Different':>10} {'Same':>6} {'Consistency%':>13}")
print(f"  {'-' * 75}")

for model_key in ['ft_08b', 'ft_2b', 'ft_4b']:
    if model_key not in MODELS:
        continue
    for kv in KV_CONFIGS:
        config_key = f"{model_key}_{kv['name']}"
        stats = consistency_by_config.get(config_key, {'total': 0, 'different': 0, 'same': 0})
        total = stats['total']
        if total > 0:
            pct = stats['different'] / total * 100
            config_label = f"{MODELS[model_key]['label']} + {kv['name']}"
            print(f"  {config_label:<35} {total:>6} {stats['different']:>10} {stats['same']:>6} {pct:>12.0f}%")
    print()


## 8. WorldSim Recommendation

In [ ]:
print('=' * 80)
print('  WORLDSIM TURBOQUANT RECOMMENDATION')
print('=' * 80)

for model_key in ['ft_08b', 'ft_2b', 'ft_4b']:
    if model_key not in MODELS:
        continue

    label = MODELS[model_key]['label']
    baseline = [r for r in RESULTS if r['model'] == model_key and r['kv_config'] == 'q8_0 (baseline)']
    turbo4 = [r for r in RESULTS if r['model'] == model_key and r['kv_config'] == 'turbo4']
    turbo3 = [r for r in RESULTS if r['model'] == model_key and r['kv_config'] == 'turbo3']

    if not baseline:
        print(f"\n  {label}: NO BASELINE DATA")
        continue

    b_score = sum(r['grades']['score'] for r in baseline) / len(baseline)
    b_json = sum(1 for r in baseline if r['grades']['json_valid']) / len(baseline) * 100
    b_lat = sum(r['latency_ms'] for r in baseline) / len(baseline)

    print(f"\n  {label}:")
    print(f'    Baseline (q8_0): score={b_score:.1f}/4, json={b_json:.0f}%, latency={b_lat:.0f}ms')

    for name, data in [('turbo4', turbo4), ('turbo3', turbo3)]:
        if not data:
            print(f'    {name}: NO DATA (server may have failed)')
            continue

        t_score = sum(r['grades']['score'] for r in data) / len(data)
        t_json = sum(1 for r in data if r['grades']['json_valid']) / len(data) * 100
        t_lat = sum(r['latency_ms'] for r in data) / len(data)

        score_ok = (b_score - t_score) < 0.2
        json_ok = (b_json - t_json) < 2

        if score_ok and json_ok:
            verdict = 'SAFE'
        elif score_ok or json_ok:
            verdict = 'DEGRADED'
        else:
            verdict = 'BROKEN'

        speedup = (b_lat - t_lat) / b_lat * 100 if b_lat else 0
        print(
            f'    {name}: score={t_score:.1f}/4 ({b_score - t_score:+.1f}), '
            f'json={t_json:.0f}%, latency={t_lat:.0f}ms ({speedup:+.1f}%) -> {verdict}'
        )

print('\n  Note: TurboQuant compresses KV cache, not model weights.')
print('  Biggest impact is on longer contexts (Tier 3: ctx 4096).')
print('  For short contexts (Tier 1: ctx 1024), benefit is minimal.')
print('  Sub-1B models should be reviewed carefully if turbo3 degrades JSON or personality consistency.')


## 9. Save Results

In [ ]:
results_path = REPO_ROOT / 'outputs' / 'benchmarks' / 'turboquant_kv_benchmark.json'
results_path.parent.mkdir(parents=True, exist_ok=True)

save_data = {
    'metadata': {
        'turboquant_fork': 'TheTom/llama-cpp-turboquant',
        'kv_configs': [kv['name'] for kv in KV_CONFIGS],
        'models': {k: v['label'] for k, v in MODELS.items()},
        'total_results': len(RESULTS),
    },
    'bench_results': BENCH_RESULTS,
    'speed_summary': SPEED_SUMMARY,
    'results': [{k: v for k, v in row.items() if k != 'parsed'} for row in RESULTS],
}

with open(results_path, 'w', encoding='utf-8') as handle:
    json.dump(save_data, handle, indent=2, ensure_ascii=False)

print(f'Saved: {results_path} ({results_path.stat().st_size / 1024:.0f} KB)')
